# VoiceForge Phase 6 · GPU precision
In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run the cells in order. FP32 is the reference. Reduced precision runs use CUDA autocast, and unsupported or failed modes are recorded. The final cell downloads the report, WAV samples, and GPU utilization log.

The first cell must list an NVIDIA GPU before setup continues. If connected to a local runtime, switch to a hosted Colab runtime for this benchmark.

In [ ]:
import os
import shutil
import subprocess
import tempfile

smi = shutil.which("nvidia-smi")
if not smi:
    raise RuntimeError("No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell.")
subprocess.run([smi, "-L"], check=True)

%pip -q install uv
os.environ["VOICEFORGE_PHASE6_RESULTS"] = tempfile.mkdtemp(prefix="voiceforge-phase6-", dir="/content")

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
if ! command -v espeak-ng >/dev/null; then
  apt-get -qq update
  apt-get -qq install -y espeak-ng
fi
repo=/content/voiceforge-precision
if [[ ! -d "$repo/.git" ]]; then
  git clone --branch voiceforge-gpu-precision https://github.com/dne-labs/Voice-inference.git "$repo"
fi
cd "$repo/services/inference"
uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  voiceforge-synthesize --device cuda --output /tmp/voiceforge-phase6-warmup.wav

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
cd /content/voiceforge-precision/services/inference
nvidia-smi --query-gpu=timestamp,name,utilization.gpu,memory.used --format=csv -l 1 > "$VOICEFORGE_PHASE6_RESULTS/gpu-utilization.csv" &
poller=$!
trap 'kill "$poller" 2>/dev/null || true' EXIT
HF_HUB_OFFLINE=1 OMP_NUM_THREADS=2 uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  python ../../experiments/05_precision/run.py --rounds 3 --output "$VOICEFORGE_PHASE6_RESULTS/precision.json"

In [ ]:
from pathlib import Path
import json
import os
import shutil
from google.colab import files

results = Path(os.environ["VOICEFORGE_PHASE6_RESULTS"])
report = json.loads((results / "precision.json").read_text())
print(report["setup"])
for run in report["runs"]:
    print(run["label"], run["status"], run.get("p50_first_pcm_ms"), run.get("reason", ""))
archive = shutil.make_archive(str(results), "zip", results)
files.download(archive)